Import needed libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import animation
from IPython.display import HTML, display
import ipywidgets as widgets
from matplotlib.animation import PillowWriter
import os
import base64

mpl.rcParams['animation.embed_limit'] = 300  # MB

## Game Rules
We use the following rules for the "Game of Life", given that each cell has 8 neighbors:
1) Cells with fewer than 2 neighbors die if they were alive and stay dead if were already dead
2) Cells with more than 3 neighbors die if they were alive and stay dead if were already dead
3) Alive cells with exactly 2 or 3 neighbors stay alive
4) Dead cells with exactly 3 neighbors become alive

To evolve the world we use the properties of numpy arrays to translate the grid representing the world in each one of the 8 directions where the neighbors are with respect to one cell (left, right, up, down and the four diagonals). For each translation we obtain a grid having in each cell the state of corresponding cell's neighbor, and by summing them we obtain a grid containing the count of each cell's neighbors. \
For the boundary condition we use a "wrap boundary condition" which means that the world has a toroidal shape.

In [ ]:
def evolution(world, neighbours):
    new_world = np.zeros_like(world)
    
    survive = (world == 1) & (neighbours == 2)
    born = (neighbours == 3)

    new_world[survive | born] = 1
    return new_world


def count_neighbors(world):
    return (
        np.roll(world,1,0) + np.roll(world,-1,0) +
        np.roll(world,1,1) + np.roll(world,-1,1) +
        np.roll(np.roll(world,1,0),1,1) +
        np.roll(np.roll(world,1,0),-1,1) +
        np.roll(np.roll(world,-1,0),1,1) +
        np.roll(np.roll(world,-1,0),-1,1)
    )

## Game of Life Class

In [ ]:
class GameOfLife:
    
    def __init__(self, N, initial_state=None):
        self.N = N
        
        if initial_state is None:
            self.world = np.zeros((N, N), dtype=int)
        else:
            self.world = initial_state.copy()

        self.initial_state = self.world.copy()
        self.population_history = []
        self.birth_history = [0]
        self.death_history = [0]
        self.activity_map = np.zeros_like(self.world)

    def step(self):

        neighbours = count_neighbors(self.world)
        new_world = evolution(self.world, neighbours)
        previous_world = self.world.copy()

        # Born cells: were 0, become 1
        born = np.sum((self.world == 0) & (new_world == 1))

        # Dead cells: were 1 become 0
        dead = np.sum((self.world == 1) & (new_world == 0))

        # Population
        p = np.sum(self.world) / (self.N**2)
    
        self.world = new_world

        self.population_history.append(p)
        self.birth_history.append(born / (self.N**2))
        self.death_history.append(dead / (self.N**2))
        self.activity_map += (previous_world != self.world)


## Predefined Patterns

In [ ]:
def pattern_glider():
    return np.array([
        [0,1,0],
        [0,0,1],
        [1,1,1]
    ])

def pattern_blinker():
    return np.array([
        [1,1,1]
    ])

def pattern_block():
    return np.array([
        [1,1],
        [1,1]
    ])

def pattern_beehive():
    return np.array([
        [0,1,1,0],
        [1,0,0,1],
        [0,1,1,0]
    ])

def pattern_loaf():
    return np.array([
        [0,1,1,0],
        [1,0,0,1],
        [0,1,0,1],
        [0,0,1,0]
    ])

def pattern_boat():
    return np.array([
        [1,1,0],
        [1,0,1],
        [0,1,0]
    ])

def pattern_tub():
    return np.array([
        [0,1,0],
        [1,0,1],
        [0,1,0]
    ])

def pattern_toad():
    return np.array([
        [0,1,1,1],
        [1,1,1,0]
    ])

def pattern_beacon():
    return np.array([
        [1,1,0,0],
        [1,1,0,0],
        [0,0,1,1],
        [0,0,1,1]
    ])

def pattern_lwss():
    return np.array([
        [0,1,1,1,1],
        [1,0,0,0,1],
        [0,0,0,0,1],
        [1,0,0,1,0]
    ])

def pattern_mwss():
    return np.array([
        [0,0,1,0,0,0],
        [1,0,0,0,1,0],
        [0,0,0,0,0,1],
        [1,0,0,0,0,1],
        [0,1,1,1,1,1]
    ])

def pattern_hwss():
    return np.array([
        [0,0,1,1,0,0,0],
        [1,0,0,0,0,1,0],
        [0,0,0,0,0,0,1],
        [1,0,0,0,0,0,1],
        [0,1,1,1,1,1,1]
    ])

def pattern_r_pentomino():
    return np.array([
        [0,1,1],
        [1,1,0],
        [0,1,0]
    ])

def pattern_acorn():
    return np.array([
        [0,1,0,0,0,0,0],
        [0,0,0,1,0,0,0],
        [1,1,0,0,1,1,1]
    ])

def pattern_diehard():
    return np.array([
        [0,0,0,0,0,0,1,0],
        [1,1,0,0,0,0,0,0],
        [0,1,0,0,0,1,1,1]
    ])


def pattern_random(N, density=0.3):
    return (np.random.rand(N,N) < density).astype(int) 

def pattern_pulsar():
    # Pulsar oscillator
    # Dimension: 13x13
    
    return np.array([
        [0,0,1,1,1,0,0,0,1,1,1,0,0],
        [0,0,0,0,0,0,0,0,0,0,0,0,0],
        [1,0,0,0,0,1,0,1,0,0,0,0,1],
        [1,0,0,0,0,1,0,1,0,0,0,0,1],
        [1,0,0,0,0,1,0,1,0,0,0,0,1],
        [0,0,1,1,1,0,0,0,1,1,1,0,0],
        [0,0,0,0,0,0,0,0,0,0,0,0,0],
        [0,0,1,1,1,0,0,0,1,1,1,0,0],
        [1,0,0,0,0,1,0,1,0,0,0,0,1],
        [1,0,0,0,0,1,0,1,0,0,0,0,1],
        [1,0,0,0,0,1,0,1,0,0,0,0,1],
        [0,0,0,0,0,0,0,0,0,0,0,0,0],
        [0,0,1,1,1,0,0,0,1,1,1,0,0]
    ])

def glider_gun():
    return np.array([
        [0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0],
        [0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0],
        [0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,1,1],
        [0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,1,1],
        [1,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0],
        [1,1,0,0,0,0,0,0,0,0,1,0,0,0,1,0,1,1,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0],
        [0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0],
        [0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],
        [0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0]
    ])

## Advanced Animation

In [ ]:
def interactive_animation(world, steps, interval):

    game = GameOfLife(N, world)

    fig = plt.figure(figsize=(14,14))

    gs = fig.add_gridspec(2, 2)

    ax1 = fig.add_subplot(gs[0, 0])  # World
    ax3 = fig.add_subplot(gs[0, 1])  # Heatmap
    ax2 = fig.add_subplot(gs[1, :])  # Population graph (spans both columns)
    
    im = ax1.imshow(game.world, cmap='binary')
    title = ax1.set_title("")
    ax1.set_axis_off()

    pop_line, = ax2.plot([], [], label="Population", color="black")
    birth_line, = ax2.plot([], [], label="Born", color="green")
    death_line, = ax2.plot([], [], label="Dead", color="red")

    ax2.set_title("Population Dynamics")
    ax2.set_xlabel("Generation")
    ax2.set_ylabel("Cells percentage")

    ax2.margins(y=0.05)
    
    # handle legends
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax2.legend(lines2, labels2)

    heat = ax3.imshow(game.activity_map, cmap='hot')
    ax3.set_title("Cell Activity")
    ax3.set_axis_off()

    # Initial State
    game.population_history = [np.sum(game.world) / game.N**2]
    generation = {"value": 0}

    running = {"value": True}

    def update(frame):

        # Evolve only when going on
        if running["value"] and generation["value"] < frame:
            game.step()
            generation["value"] += 1

        # Update image
        im.set_data(game.world)

        # Update title
        current_population = np.sum(game.world)
        title.set_text(
            f"Generation: {generation['value']}  |  "
            f"Population: {current_population}"
        )

        # Update graph
        pop_line.set_data(
            range(len(game.population_history)),
            game.population_history
        )

        birth_line.set_data(
            range(len(game.birth_history)),
            game.birth_history
        )

        death_line.set_data(
            range(len(game.death_history)),
            game.death_history
        )

        # Dynamic axes
        ax2.relim()
        ax2.autoscale_view()

        heat.set_data(game.activity_map)
        heat.set_clim(0, np.max(game.activity_map) + 1)

        return [im, heat, pop_line, birth_line, death_line, title]


    anim = animation.FuncAnimation(
        fig,
        update,
        frames=steps,
        interval=interval,
        blit=True
    )
    
    plt.close(fig)
    
    # anim.save("pattern_gifs/acorn.gif", writer=PillowWriter(fps=18), dpi=100)
    display(HTML(anim.to_jshtml()))


## Simulation Parameters

We can use few parameters to define the initial configuration of the world and the characteristics of the animation. These are `N`, which is the size of the grid NxN, `steps` which determins how many iterations we want to compute for the time evolution, `interval` is a dedicated parameter for the animation (how much time to wait between frames expressed in milliseconds), `pattern_type` which determins what initial configuration we want to use among the once encoded, and last `density` is a parameter that affects only the population density in case we used `pattern_type = 'random'`. 

Here is a list of all the encoded patterns and initial configurations:
#### 1) Static patterns
- `block`
- `beehive`
- `loaf`
- `boat`
- `tub`
#### 2) Oscillator patterns
- `blinker`
- `toad`
- `beacon`
- `pulsar`
#### 3) Spaceship patterns
- `glider`
- `lwss` (light weight spaceship)
- `mwss` (middle weight spaceship)
- `hwss` (heavy weight spaceship)
### 4) Advanced patterns
- `r_pentomino`
- `acorn`
- `diehard`
- `glider_gun`

### Logic Ports
Using the `glider_gun` pattern it is possible to produce logic ports as it guarantees a periodic signal which is made out of the produced gliders. Morover, one of their properties is that when they collide they annihilate each other.

#### Implemented Logic Ports
- `NOT` –> 2 `glider_gun`s are used, one as signal and the other as logic port; the gliders of the first collide with the once produced by the second, so the signal is destroyed (converted from 1 to 0), while if the input is turned off we have the stream of gliders of the logic port (conversion from 0 to 1). We obtain the truth table of the NOT gate.
- `AND` –> use 3 `glider_gun`s, 2 as input signals A and B and 1 as logic gate; two of them are placed up, one next to each other horizontaly, and the third is placed down aligned with the first; we notice that we can read the signal from the second gun if and only if both of them are switched on, thus obtaining the truth table of the AND gate.
- `OR` –> use 4 `glider_gun`s, 2 as input signals A and B and 2 as logic gates; they are placed one on top of each other and we use the 2 central guns as inputs and the two external once as logic port; we read the output from the upper one which happens to be off only when the central guns are off, thus obtaining the truth table of a OR gate.


In [ ]:
N = 150
steps = 1000
interval = 80
pattern_type = "acorn"
density = 0.3 # Only in random

## World Building

In [ ]:
world = np.zeros((N, N), dtype=int)

def place_pattern(world, pattern):
    # Automatically centers the pattern inside the grid
    px, py = pattern.shape
    cx = (N - px) // 2
    cy = (N - py) // 2
    world[cx:cx+px, cy:cy+py] = pattern
    return world


# Pattern dictionary
patterns = {
    "glider": pattern_glider, # Spaceship
    "blinker": pattern_blinker, # Oscillator
    "block": pattern_block, # Static
    "beehive": pattern_beehive, # Static
    "loaf": pattern_loaf, # Static
    "boat": pattern_boat, # Static
    "tub": pattern_tub, # Static
    "toad": pattern_toad, # Oscillator
    "beacon": pattern_beacon, # Oscillator
    "lwss": pattern_lwss, # Spaceship
    "mwss": pattern_mwss, # Spaceship
    "hwss": pattern_hwss, # Spaceship
    "r_pentomino": pattern_r_pentomino,
    "acorn": pattern_acorn,
    "diehard": pattern_diehard,
    "pulsar": pattern_pulsar, # Oscillator
    "glider_gun": glider_gun
}


if pattern_type == "random":
    world = pattern_random(N, density)

elif pattern_type == "custom":
    world[30,30] = 1
    world[31,31] = 1
    world[32,29:32] = 1

# Single patterns implementation
elif pattern_type in patterns:
    world = place_pattern(world, patterns[pattern_type]())

elif pattern_type == "NOT":
    gunA = patterns["glider_gun"]()
    gunB = patterns["glider_gun"]()[::-1]
    world[20:29,20:56] = gunA
    world[150:159,20:56] = gunB

elif pattern_type == "AND":
    gunA = patterns["glider_gun"]()
    gunB = patterns["glider_gun"]()
    gunOUT = patterns["glider_gun"]()[::-1]
    world[20:29,20:56] = gunA
    world[20:29,80:116] = gunB
    world[150:159,20:56] = gunOUT

elif pattern_type == "OR":
    gunA = patterns["glider_gun"]()
    gunB = patterns["glider_gun"]()
    gun = patterns["glider_gun"]()
    gunOUT = patterns["glider_gun"]()[::-1]
    world[70:79,20:56] = gunA
    world[120:129,20:56] = gunB
    world[20:29,20:56] = gun
    world[180:189,20:56] = gunOUT

else:
    raise ValueError(f"Pattern '{pattern_type}' not recognized.")

## Start Animation

In [ ]:
interactive_animation(world, steps, interval)

In [ ]:
# Function to display gifs
def show_gif(gif_info_list, gif_folder="."):
    
    html_content = """
    <div style="display: flex; flex-wrap: wrap; gap: 40px;">
    """

    for filename, title, description in gif_info_list:
        
        path = os.path.join(gif_folder, filename)
        
        if not os.path.exists(path):
            print(f"File non trovato: {path}")
            continue
        
        # Codifica GIF in base64 per incorporarla nell'HTML
        with open(path, "rb") as f:
            encoded = base64.b64encode(f.read()).decode("utf-8")
        
        html_content += f"""
        <div style="text-align: center;">
            <h3>{title}</h3>
            <img src="data:image/gif;base64,{encoded}" width="250"><br>
            <p style="max-width:250px;">{description}</p>
        </div>
        """

    html_content += "</div>"
    
    display(HTML(html_content))

# Pattern Description
In the Game of Life, patterns are all devided into six classes each with its own characteristics. Four of them are stable, namely still lifes, oscillators, spaceships and glider guns; the other two are unstable and contain all objects that cannot be classified in the previous four classes.

## Stable Classes
As the name says these classes contain patterns that do not die when left alone, unperturbated by external factors. In the following we give a brief description of each one of them to highlight their properties.

#### 1) Still life
Still lives are patterns that remain completely unchanged from one generation to the next, so we can say they have period 1. Common still lifes are: block, beehive, loaf, boat, tub.

In [ ]:
gif_SL = [
    ("pattern_block.gif", "Block", "The smallest still life pattern."),
    ("pattern_beehive.gif", "Beehive", "A stable six-cell still life."),
    ("pattern_loaf.gif", "Loaf", "An asymmetric still life."),
    ("pattern_boat.gif", "Boat", "A compact five-cell still life."),
    ("pattern_tub.gif", "Tub", "A symmetric four-cell still life.")
]
show_gif(gif_SL, 'pattern_gifs')

#### 2) Oscillator
Oscillators are patterns that return to their original configuration after a fixed number of generations. Therefore they have a period greater than one, but they do not move across the grid. Common examples are the blinker, the toad, the beacon, the pulsar.

In [ ]:
gif_O = [
    ("pattern_blinker.gif", "Blinker", "The simplest oscillator (period 2). A line of three cells that alternates between horizontal and vertical orientations."),
    ("pattern_toad.gif", "Toad", "A period-2 oscillator made of six cells arranged in two offset rows that flip back and forth each generation."),
    ("pattern_beacon.gif", "Beacon", "A period-2 oscillator formed by two adjacent blocks that cyclically “blink” at their corners."),
    ("pattern_pulsar.gif", "Pulsar", "A large period-3 oscillator composed of 48 cells. It expands and contracts rhythmically in a highly symmetric pattern.")
]
show_gif(gif_O, 'pattern_gifs')

#### 3) Spaceship
Spaceships are patterns that translate themselves across the grid while periodically returning to their original shape. They demonstrate how simple local interaction rules can produce sustained motion. Common examples are the glider and the light, middle and high weight spaceships.

In [ ]:
gif_S = [
    ("pattern_glider.gif", "Glider", "The smallest and most famous spaceship. A five-cell pattern that moves diagonally across the grid with period 4."),
    ("pattern_lwss.gif", "Lightweight spaceship", "A small orthogonal spaceship that travels horizontally across the grid. It has period 4 and is the smallest of the classic spaceships."),
    ("pattern_mwss.gif", "Middleweight spaceship", "A larger version of the LWSS. It moves horizontally with period 4 and has a wider structure."),
    ("pattern_hwss.gif", "Highweight spaceship", "The largest of the basic spaceships. It also travels horizontally with period 4 but contains more cells and a broader shape.")
]
show_gif(gif_S, 'pattern_gifs')

#### 4) Glider gun
Although they create infinite growth, these patterns are structurally stable and periodic. They do not evolve chaotically, instead they repeatedly emit moving structures.


In [ ]:
gif_GG = [
    ("glider_gun.gif", "Glider gun", "The core pattern is stable and repeats indefinitely, but it produces unbounded growth by generating an infinite stream of spaceships.")
]
show_gif(gif_GG, 'pattern_gifs')

## Unstable Classes
Unstable patterns evolve dynamically, often producing growth, fragmentation, or complex interactions before reaching a final state, if they reach one at all. For this reason, they can be devided into two categories: predictable and unpredictable.

#### 1) Predictable unstable patterns
These patterns show complex or chaotic growth, but their long-term behavior is known: they eventually stabilize into a finite combination of still lifes, oscillators, and sometimes spaceships, or either they die. Below we have the example of the acorn.

In [ ]:
gif_GG = [
    ("acorn.gif", "Acorn", "A seven-cell pattern that generates large chaotic expansion before settling into a complex but finite configuration.")
]
show_gif(gif_GG, 'pattern_gifs')

#### 2) Unpredictable unstable patterns
Like predictable once, these patterns show cmplex and chaotic growth, but there is no evidence that they will eventually end.